# A Simple Calculator

This file shows the implementation of a *symbolic calculator* using the scanner generator `ocamllex` and the
parser generator `Menhir`.  (The Python version of this notebook uses the library `lark`.)  The grammar for the
language implemented by this parser is as follows:
$$
\begin{array}{lcl}
  \texttt{stmnt}   & \rightarrow & \;\texttt{IDENTIFIER}\; \texttt{':='} \; \;\texttt{expr}\; \texttt{';'} \\
                   & \mid        & \;\texttt{expr}\; \texttt{';'} \\[0.2cm]
  \texttt{expr}    & \rightarrow & \;\texttt{expr}\; \texttt{'+'} \; \texttt{product}  \\
                   & \mid        & \;\texttt{expr}\; \texttt{'-'} \; \texttt{product}  \\
                   & \mid        & \;\texttt{product}                                  \\[0.2cm]
  \texttt{product} & \rightarrow & \;\texttt{product}\; \texttt{'*'} \;\texttt{factor} \\
                   & \mid        & \;\texttt{product}\; \texttt{'/'} \;\texttt{factor} \\
                   & \mid        & \;\texttt{factor}                                   \\[0.2cm]
  \texttt{factor}  & \rightarrow &   \texttt{'('} \; \texttt{expr} \;\texttt{')'}      \\
                   & \mid        & \;\texttt{NUMBER}                                   \\
                   & \mid        & \;\texttt{IDENTIFIER}
\end{array}
$$

## Setup and Imports

The file `../lib/parser_generator.ml` provides the function `generate_parser`, which runs `ocamllex` and
`Menhir`, and the module `Parse_tree`, which describes parse trees in the style of `lark`: a parse tree is either
a node `Tree (name, children)` or a leaf `Token t`, where the token `t` knows its type `t.kind` and its value `t.value`.

In [ ]:
#use "../lib/nb.ml";;
load_lib "parser_generator.ml";;
open Parse_tree

## Specification of the Grammar

`lark` defines the tokens and the grammar rules in a single string.  With `ocamllex` and `Menhir`, the scanner and
the parser are specified separately: the *scanner specification* defines the tokens via regular expressions, and the
*grammar* defines the structure.  Let us break down the components of our grammar definition:

* `%token`: Every token has to be declared.  The tokens `IDENTIFIER` and `NUMBER` carry a value of type
  `Parse_tree.tok`, which stores the string that has been recognized.  The other tokens carry no value; they are the
  counterparts of the *anonymous* tokens like `"+"` of the `lark` grammar.  We have named them like the tokens that
  `lark` creates for these strings, e.g. `PLUS` and `LPAR`.
* `%start <Parse_tree.tree> start`: This tells Menhir that the entry point of our grammar is the variable `start`, and
  that the parser returns a parse tree.  The token `EOF` marks the end of the input.  The declaration `%type` states
  that the other variables produce parse trees, too.
* `stmnt`, `expr`, `product`, `factor`: These variables define the hierarchical structure of our mathematical expressions,
   natively enforcing operator precedence (e.g., multiplication in `product` happens before addition in `expr`).
* **Semantic actions**: Every grammar rule is followed by an OCaml expression in braces that computes the result of
  the rule.  Here, the result is a node of the parse tree.  `e = expr` gives the name `e` to the result of `expr`.
  - In `lark`, an *alias* like `-> add` names the node of the parse tree.  In Menhir, the action
    `{ Tree ("add", [e; p]) }` creates a node with the name `add`.
  - In `lark`, the prefix `?` *inlines* a rule that has only one child.  In Menhir, the action `{ p }` simply returns
    the child.
  - The anonymous tokens like `"+"` do not show up in the parse tree, since the actions do not mention them.

The scanner specification for `ocamllex` consists of a *header* in braces, which contains OCaml code, some
definitions of regular expressions, and the rule `token`.  Every case of this rule consists of a regular expression
and an action that returns the token that has been recognized.
* `cname` and `number` correspond to the terminals `common.CNAME` and `common.NUMBER` that are imported by the `lark`
  grammar: `cname` describes standard variable names, while `number` handles both integers and floating-point numbers.
* White space is skipped by calling the rule `token` recursively.  This corresponds to `%ignore WS`.
* `Parse_tree.token "NUMBER" lexbuf` creates a token of the type `NUMBER` from the string that has just been
  recognized.  It also stores the line and the column of the token.
* A character that is not covered by any of the regular expressions raises the exception `Unexpected_input`.

In [ ]:
let calc_lexer = {|
{
  open Calc_parser
}

let digit  = ['0'-'9']
let int    = digit+
let cname  = ['_' 'a'-'z' 'A'-'Z'] ['_' 'a'-'z' 'A'-'Z' '0'-'9']*
let number = int ['e' 'E'] ['+' '-']? int
           | (int '.' int? | '.' int) (['e' 'E'] ['+' '-']? int)?
           | int

rule token = parse
  | [' ' '\t' '\r' '\012']+ { token lexbuf }                     (* ignore whitespace *)
  | '\n'                    { Lexing.new_line lexbuf; token lexbuf }
  | ":="                    { ASSIGN }
  | ';'                     { SEMI }
  | '+'                     { PLUS }
  | '-'                     { MINUS }
  | '*'                     { STAR }
  | '/'                     { SLASH }
  | '('                     { LPAR }
  | ')'                     { RPAR }
  | number                  { NUMBER (Parse_tree.token "NUMBER" lexbuf) }
  | cname                   { IDENTIFIER (Parse_tree.token "IDENTIFIER" lexbuf) }
  | eof                     { EOF }
  | _                       { raise (Parse_tree.unexpected lexbuf ("No terminal matches " ^ Lexing.lexeme lexbuf)) }
|}

In [ ]:
let calc_grammar = {|
%{
  open Parse_tree
%}

%token <Parse_tree.tok> IDENTIFIER NUMBER
%token ASSIGN SEMI PLUS MINUS STAR SLASH LPAR RPAR EOF

%start <Parse_tree.tree> start
%type  <Parse_tree.tree> stmnt expr product factor

%%

start: s = stmnt EOF                          { s }

stmnt: id = IDENTIFIER ASSIGN e = expr SEMI  { Tree ("assign", [Token id; e]) }
     | e = expr SEMI                          { Tree ("expr_stmnt", [e]) }

expr: e = expr PLUS p = product               { Tree ("add", [e; p]) }
    | e = expr MINUS p = product              { Tree ("sub", [e; p]) }
    | p = product                             { p }

product: p = product STAR f = factor          { Tree ("mul", [p; f]) }
       | p = product SLASH f = factor         { Tree ("div", [p; f]) }
       | f = factor                           { f }

factor: LPAR e = expr RPAR                    { e }
      | n = NUMBER                            { Tree ("number", [Token n]) }
      | id = IDENTIFIER                       { Tree ("var", [Token id]) }
|}

We generate the scanner and the parser.  The function `generate_parser` writes the scanner specification to the file
`_build/calc_lexer.mll` and the grammar to the file `_build/calc_parser.mly`.  Then it runs `ocamllex` and
`menhir --lalr`, which creates an LALR(1) parser, like the option `parser='lalr'` of `lark`.  Finally, it loads the
generated modules `Calc_lexer` and `Calc_parser`.

In [ ]:
generate_parser ~name:"calc" ~lexer:calc_lexer ~grammar:calc_grammar

The function `parse` parses a string and returns the parse tree.  It corresponds to the method `parse` of the
`lark` parser.  `Calc_parser.start` is the parser for the start symbol `start` and `Calc_lexer.token` is the scanner.

In [ ]:
let parse (s : string) : tree =
  parse_string Calc_parser.start Calc_lexer.token s
;;
parse "x := y + z;"

## Generating the Abstract Syntax Tree (ADT)

Rather than evaluating the parse tree directly, we convert it into an abstract data type (ADT).  In Python, this ADT
consists of nested tuples.  For example, the string `x := y + z;` is parsed into the ADT
`(':=', 'x', ('+', ('var', 'y'), ('var', 'z')))`.  In OCaml, we define the variant types `op`, `expr`, and `stmnt`
instead.  The string `x := y + z;` is parsed into the ADT
```
Assign ("x", BinOp (Add, Var "y", Var "z"))
```
**Type safety:** The Python function `ast_to_tuples` can return arbitrary tuples, and the function `evaluate_adt`
needs a final case that raises an error if it encounters an unknown structure.  In OCaml, the type of an ADT
determines which structures are possible, and the compiler checks that the functions handle all of them.  Since
statements and expressions have different types, there is one conversion function for each of them.

`operator_map` maps the names of the nodes of the parse tree to the operators.

In [ ]:
type op = Add | Sub | Mul | Div

type expr =
  | Num   of float
  | Var   of string
  | BinOp of op * expr * expr

type stmnt =
  | Assign of string * expr
  | Eval   of expr

let operator_map : (string * op) list = [ ("add", Add); ("sub", Sub); ("mul", Mul); ("div", Div) ]

let rec ast_to_expr (node : tree) : expr =
  match node with
  | Tree ("number", [ Token t ]) -> Num (float_of_string t.value)
  | Tree ("var", [ Token t ]) -> Var t.value
  | Tree (name, [ left; right ]) when List.mem_assoc name operator_map ->
      BinOp (List.assoc name operator_map, ast_to_expr left, ast_to_expr right)
  | _ -> invalid_arg ("ast_to_expr: unexpected node " ^ pretty node)

let ast_to_stmnt (node : tree) : stmnt =
  match node with
  | Tree ("assign", [ Token var; e ]) -> Assign (var.value, ast_to_expr e)
  | Tree ("expr_stmnt", [ e ]) -> Eval (ast_to_expr e)
  | _ -> invalid_arg ("ast_to_stmnt: unexpected node " ^ pretty node)

In [ ]:
ast_to_stmnt (parse "x := y + z;")

## Evaluating the Abstract Syntax Tree

Now that we have our ADT, we can write our own recursive evaluation functions.  The variable `names2Values` stores
the values associated with the variables we define.  In Python, it is a global dictionary that is changed by the
assignments.  In OCaml, it is a reference to an (immutable) dictionary of type `Dict.t`, which is replaced by an
extended dictionary on every assignment.

In [ ]:
let names2Values : (string, float) Dict.t ref = ref Dict.empty

The function `evaluate_expr` computes the value of an expression, while the function `evaluate_adt` executes a
statement: an assignment changes `names2Values`, while an expression statement prints the value of the expression.

In [ ]:
let rec evaluate_expr (adt : expr) : float =
  match adt with
  | Num value -> value
  | Var var_name -> (
      match Dict.find_opt var_name !names2Values with
      | Some value -> value
      | None ->
          Printf.printf "Syntax Error: Variable '%s' is undefined.\n%!" var_name;
          0.0)
  | BinOp (op, left, right) -> (
      let val_left  = evaluate_expr left in
      let val_right = evaluate_expr right in
      match op with
      | Add -> val_left +. val_right
      | Sub -> val_left -. val_right
      | Mul -> val_left *. val_right
      | Div -> val_left /. val_right)

let evaluate_adt (adt : stmnt) : unit =
  match adt with
  | Assign (var_name, expr) -> names2Values := Dict.add var_name (evaluate_expr expr) !names2Values
  | Eval expr -> print_endline (float_repr (evaluate_expr expr))

## Execution

The REPL (Read-Eval-Print Loop) takes the input string, parses it into a parse tree, converts it into our ADT, and then
evaluates it.  The loop stops when the user enters an empty line.  A syntax error raises the exception
`Unexpected_input`, which is reported together with the line and the column of the error.

The OCaml kernel of *Jupyter* can not read from the keyboard.  Therefore, the function `input` from the file
`../lib/nb.ml` returns the lines that have been passed to the function `simulate_input`.

In [ ]:
let main () : unit =
  let rec loop () =
    match input "calc> " with
    | "" -> ()
    | s ->
        (try
           (* 1. Parse the string into a parse tree *)
           let raw_tree = parse s in
           (* 2. Convert it into an ADT *)
           let adt = ast_to_stmnt raw_tree in
           (* 3. Evaluate the ADT *)
           evaluate_adt adt
         with
         | Unexpected_input { message; line; column; _ } ->
             Printf.printf "Error: %s at line %d, column %d.\n%!" message line column
         | e -> Printf.printf "Error: %s\n%!" (Printexc.to_string e));
        loop ()
  in
  loop ()

In [ ]:
simulate_input [ "x := 1 + 2 * 3;"; "y := x / 2;"; "x * y - 4;"; "z + 1;"; "(1 + 2;"; "" ];;
main ()

Check the populated memory map:

In [ ]:
!names2Values

We can even get the parser states.  When it is called with the option `--dump`, Menhir writes a description of the
automaton of the generated parser to the file `_build/calc_parser.automaton`.  For every state, this file lists the
*items*, i.e. the marked grammar rules, of the state.  The tokens in square brackets are the *lookahead* tokens of a
rule.  Furthermore, the file lists the transitions and the reductions of every state.

The Python version of this notebook computes the states with the help of the custom library `formal_methods_viz`,
which inspects the internal data structures of `lark`.  This is not necessary with Menhir.  Note that Menhir adds the
item `start' -> . start` for the augmented grammar, and that the numbering of the states differs from `lark`.

In [ ]:
let automaton_file (name : string) : string = Filename.concat "_build" (name ^ "_parser.automaton")

In [ ]:
let print_lalr1_states (name : string) : unit = print_string (read_file (automaton_file name)); flush stdout

In [ ]:
print_lalr1_states "calc"